In [10]:
import openai
from llama_index.core import VectorStoreIndex, StorageContext, SimpleDirectoryReader
from llama_index.vector_stores.milvus import MilvusVectorStore
import os
from PIL import Image
import mimetypes

In [2]:
openai.api_key = "sk-proj-"

In [5]:
milvus_folder = "memory/milvus_rag"
uri = os.path.join(milvus_folder, "milvus_demo.db")
vector_store = MilvusVectorStore(
    uri=uri, dim=1536, overwrite=False
)

In [ ]:
# load single document
# documents = SimpleDirectoryReader(
#     input_files=["./data/paul_graham_essay.txt"]
# ).load_data()

# # load multiple documents
# reader = SimpleDirectoryReader(input_dir="cases/GoldMiner/")
# docs = reader.load_data()
# print(f"Loaded {len(docs)} docs")

In [14]:
# OpenAI Embedding API call - requires 0.28 openai
def get_text_embedding(text):
    response = openai.Embedding.create(
        model="text-embedding-ada-002",
        input=text
    )
    return response['data'][0]['embedding']

# Process text files
def process_text_file(filepath):
    with open(filepath, 'r') as file:
        text_data = file.read()
    return get_text_embedding(text_data)

# Process images (manual or simple descriptions)
def process_image_file(filepath):
    # You could add a more advanced image-to-text description generator here if needed
    file_name = os.path.basename(filepath)
    image_description = f"This is an image from the game assets: {file_name}"
    return get_text_embedding(image_description)

# Function to handle each file based on its type
def process_file(filepath):
    mime_type, _ = mimetypes.guess_type(filepath)
    if mime_type:
        if mime_type.startswith('text'):
            # Process text files
            embedding = process_text_file(filepath)
        elif mime_type.startswith('image'):
            # Process image files
            embedding = process_image_file(filepath)
        else:
            # Skip unsupported file types
            print(f"Unsupported file type: {filepath}")
            return None, None
        return embedding, os.path.basename(filepath)
    else:
        print(f"Could not determine file type: {filepath}")
        return None, None

# Insert embeddings into your local MilvusVectorStore
def insert_into_milvus(embedding, file_name):
    if embedding is not None:
        vector_store.add_vector(embedding, metadata={"file_name": file_name})
        print(f"Inserted: {file_name}")
    else:
        print(f"Failed to insert: {file_name}")

# Main function to iterate over the folder and process each file
def process_folder(folder_path):
    # Walk through the folder structure
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file == ".DS_Store":  # Skip .DS_Store file
                continue
            print(f"====Processing: {file}=====")
            filepath = os.path.join(root, file)
            embedding, file_name = process_file(filepath)
            if embedding:
                print(f"Embedding: {embedding}")
                insert_into_milvus(embedding, file_name)
                print("inserted")




In [ ]:
# Example usage: Process the "GoldMiner" folder
process_folder("cases/GoldMiner/")

In [ ]:
# Query embedding for search
query_text = "A stone used in a game scene"
query_embedding = get_text_embedding(query_text)

# Search for similar embeddings in your local MilvusVectorStore
results = vector_store.search(query_embedding, top_k=5)

# Output the search results
for result in results:
    print(f"Match: {result['metadata']['file_name']}, Distance: {result['score']}")